In [0]:
%run ../../_shared/00_project_config

In [0]:
from pyspark.sql import functions as F


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
# Project-wide paths, schemas, supported seasons, and league mappings are
# loaded from 00_project_config.
#
# FOOTBALL_DATA_LEAGUES maps source league codes to canonical platform names:
#   E0  -> premier_league
#   D1  -> bundesliga
#   I1  -> serie_a
#   SP1 -> la_liga
#   F1  -> ligue_one
#
# Raw filenames and Bronze table names use the canonical league names.

SOURCE_SYSTEM = "football-data.co.uk"

LEAGUES = list(FOOTBALL_DATA_LEAGUES.values())


# -----------------------------------------------------------------------------
# Load football-data.co.uk CSV files into Bronze
# -----------------------------------------------------------------------------
# Each source file represents one league + season combination.
#
# Example:
#   2425/2425_premier_league.csv
#
# Bronze intentionally keeps all source business columns as strings by using:
#
#   inferSchema = False
#
# This avoids allowing Spark's CSV inference to assign different datatypes
# between seasons or leagues. Explicit analytical datatypes are applied later
# in the Silver layer.
#
# Technical metadata is added for lineage and traceability:
#   _season        - source season code, e.g. "2425"
#   _league        - canonical project league identifier
#   _ingested_at   - timestamp of this Bronze load
#   _source_file   - actual Unity Catalog Volume file that produced the row
#   _source_system - originating external data source

loaded_file_count = 0

for season in FOOTBALL_DATA_SEASONS:
    for league in LEAGUES:

        file_path = (
            f"{FOOTBALL_DATA_RAW_ROOT}/"
            f"{season}/"
            f"{season}_{league}.csv"
        )

        bronze_table = (
            f"{BRONZE_SCHEMA}."
            f"football_data_{league}"
        )

        # ---------------------------------------------------------------------
        # Read raw CSV
        # ---------------------------------------------------------------------

        source_df = (
            spark.read
            .option("header", True)
            .option("inferSchema", False)
            .csv(file_path)
        )

        # ---------------------------------------------------------------------
        # Add Bronze metadata
        # ---------------------------------------------------------------------

        bronze_df = (
            source_df
            .withColumn(
                "_season",
                F.lit(season),
            )
            .withColumn(
                "_league",
                F.lit(league),
            )
            .withColumn(
                "_ingested_at",
                F.current_timestamp(),
            )
            .withColumn(
                "_source_file",
                F.col("_metadata.file_path"),
            )
            .withColumn(
                "_source_system",
                F.lit(SOURCE_SYSTEM),
            )
        )

        # ---------------------------------------------------------------------
        # Write to Bronze Delta table
        # ---------------------------------------------------------------------
        # One Bronze table is maintained per league and contains all supported
        # seasons for that league.
        #
        # Append mode preserves ingestion history.
        #
        # mergeSchema is intentionally enabled because football-data.co.uk may
        # add or remove CSV columns between seasons. Bronze captures those
        # source changes, while Silver defines the stable analytical schema.

        (
            bronze_df.write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(bronze_table)
        )

        loaded_file_count += 1

        print(
            f"Loaded football-data.co.uk Bronze file "
            f"{loaded_file_count}: "
            f"{league} / {season} "
            f"-> {bronze_table}"
        )


# -----------------------------------------------------------------------------
# Completion message
# -----------------------------------------------------------------------------

print(
    f"Successfully loaded {loaded_file_count} "
    "football-data.co.uk source files into Bronze."
)